<a href="https://colab.research.google.com/github/shehan-jayasinghe/-generative_ai_tutorial/blob/main/langgraph_stage_03.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import sys

# Uninstall all langchain related packages to clear potential conflicts
!{sys.executable} -m pip uninstall -y langchain langchain-core langchain-openai langchain-community langgraph langgraph-prebuilt langchain-tavily pyowm

# Install the latest compatible versions
!{sys.executable} -m pip install langchain -q
!{sys.executable} -m pip install langchain-openai -q
!{sys.executable} -m pip install langchain-community -q
!{sys.executable} -m pip install langgraph -q
!{sys.executable} -m pip install langchain-tavily -q
!{sys.executable} -m pip install pyowm -qU

Found existing installation: langchain 1.3.10
Uninstalling langchain-1.3.10:
  Successfully uninstalled langchain-1.3.10
Found existing installation: langchain-core 1.4.8
Uninstalling langchain-core-1.4.8:
  Successfully uninstalled langchain-core-1.4.8
Found existing installation: langchain-openai 1.3.2
Uninstalling langchain-openai-1.3.2:
  Successfully uninstalled langchain-openai-1.3.2
Found existing installation: langchain-community 0.4.2
Uninstalling langchain-community-0.4.2:
  Successfully uninstalled langchain-community-0.4.2
Found existing installation: langgraph 1.2.6
Uninstalling langgraph-1.2.6:
  Successfully uninstalled langgraph-1.2.6
Found existing installation: langgraph-prebuilt 1.1.0
Uninstalling langgraph-prebuilt-1.1.0:
  Successfully uninstalled langgraph-prebuilt-1.1.0
Found existing installation: langchain-tavily 0.2.18
Uninstalling langchain-tavily-0.2.18:
  Successfully uninstalled langchain-tavily-0.2.18
Found existing installation: pyowm 3.5.0
Uninstalling 

In [12]:

from typing import Annotated, Sequence, TypedDict
from langchain_core.messages import BaseMessage, ToolMessage, SystemMessage
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langgraph.graph.message import add_messages
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode

In [5]:
class AgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], add_messages]



In [11]:

import os
from google.colab import userdata

os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')

In [7]:

import os
from google.colab import userdata

os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')


In [8]:

llm = ChatOpenAI(model = "gpt-4o")

In [13]:

# from langchain_tavily import TavilySearch
# from langchain_community.utilities import OpenWeatherMapAPIWrapper

@tool
def addtion(a: int, b:int):
    """This is the addition function that adds 2 numbers"""
    return a + b

@tool
def subtraction(a: int, b: int):
    """This is the subtraction function that subtract 2 numbers"""
    return a - b

@tool
def multiplication(a: int, b: int):
    """This is the multiplication function that multiply 2 numbers"""
    return a * b

@tool
def division(a: int, b: int):
    """This is the division function that divide 2 numbers"""
    return a / b

# os.environ["OPENWEATHERMAP_API_KEY"] = userdata.get('OPENWEATHERMAP_API_KEY')
# os.environ['TAVILY_API_KEY'] = userdata.get('TAVILY_API_KEY')

# # Initialize DuckDuckGo search tool
# search_tool = TavilySearch()

# # Initialize OpenWeatherMap tool
# weather = OpenWeatherMapAPIWrapper()

# weather_tool = load_tools(["openweathermap-api"], llm)[0]

In [18]:

tools = [addtion, subtraction, multiplication, division]
# tools = [search_tool, weather_tool]

In [19]:

llm_with_tools = llm.bind_tools(tools)

In [20]:

def llm_call(state:AgentState) -> AgentState:
    system_prompt = SystemMessage(content=
        "You are an intelligent AI assistant."
    )
    response = llm_with_tools.invoke([system_prompt] + state["messages"])
    return {"messages": [response]}


def decision(state: AgentState):
    messages = state["messages"]
    last_message = messages[-1]
    if not last_message.tool_calls:
        return "end"
    else:
        return "continue"

In [21]:

def llm_call(state:AgentState) -> AgentState:
    system_prompt = SystemMessage(content=
        "You are an intelligent AI assistant."
    )
    response = llm_with_tools.invoke([system_prompt] + state["messages"])
    return {"messages": [response]}


def decision(state: AgentState):
    messages = state["messages"]
    last_message = messages[-1]
    if not last_message.tool_calls:
        return "end"
    else:
        return "continue"

In [ ]:
graph = StateGraph(AgentState)

graph.add_node("agent", llm_call)
tool_node = ToolNode(tools=tools)
graph.add_node("tools", tool_node)

graph.set_entry_point("agent")
graph.add_conditional_edges(
    "agent",
    decision,
    {
        "continue": "tools",
        "end": END,
    },
)
graph.add_edge("tools", "agent")

app = graph.compile()


In [ ]:

from IPython.display import Image, display
display(Image(app.get_graph().draw_mermaid_png()))

In [ ]:
def print_stream(stream):
    for s in stream:
        message = s["messages"][-1]
        if isinstance(message, tuple):
            print(message)
        else:
            message.pretty_print()

In [ ]:

inputs = {"messages": [("user", "Add 40 and 12. Then multiply the result by 6.")]}
print_stream(app.stream(inputs, stream_mode="values"))

In [ ]:
nputs = {"messages": [("user", "what is codepro lk?")]}
print_stream(app.stream(inputs, stream_mode="values"))
